# CreditLens — Day 2 data quality and exploratory analysis

This notebook documents integrity checks and concise EDA for the approved UCI dataset. It does not train a model, resample data or select a probability threshold. The fixed test labels are used only for stratified assignment and a mechanical split-balance audit; exploratory figures use training rows only.

## Reproduce the derived artefacts

From the repository root, run `Rscript scripts/day2_prepare.R`. The script checks both SHA-256 digests before reading the workbook, preserves the raw files, creates the validated CSV and split assignments, and regenerates the summary and figures.

In [1]:
from pathlib import Path
import csv
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    load_split_assignments, load_validated_rows,
    primary_feature_names, verify_raw_checksums,
)
verify_raw_checksums(PROJECT_ROOT / 'data' / 'raw')
rows = load_validated_rows(PROJECT_ROOT / 'data' / 'processed' / 'credit_default_validated.csv')
assignments = load_split_assignments(PROJECT_ROOT / 'data' / 'processed' / 'split_assignments.csv')
summary = json.loads((PROJECT_ROOT / 'reports' / 'day2_data_quality_summary.json').read_text())
print(f"Validated {len(rows):,} rows; raw checksums match.")

Validated 30,000 rows; raw checksums match.


## Shape, fields and sample structure

In [2]:
print(summary['dataset'])
sample_fields = [name for name in summary['dataset']['column_names'] if name != 'DEFAULT_NEXT_MONTH']
[{field: row[field] for field in sample_fields[:8]} for row in rows[:3]]  # non-target sample

{'rows': 30000, 'columns': 25, 'column_names': ['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'DEFAULT_NEXT_MONTH']}


[{'ID': '1',
  'LIMIT_BAL': '20000',
  'SEX': '2',
  'EDUCATION': '2',
  'MARRIAGE': '1',
  'AGE': '24',
  'PAY_0': '2',
  'PAY_2': '2'},
 {'ID': '2',
  'LIMIT_BAL': '120000',
  'SEX': '2',
  'EDUCATION': '2',
  'MARRIAGE': '2',
  'AGE': '26',
  'PAY_0': '-1',
  'PAY_2': '2'},
 {'ID': '3',
  'LIMIT_BAL': '90000',
  'SEX': '2',
  'EDUCATION': '2',
  'MARRIAGE': '2',
  'AGE': '34',
  'PAY_0': '0',
  'PAY_2': '0'}]

## Integrity, missingness and duplicate profiles

In [3]:
{key: summary[key] for key in [
    'id_unique', 'exact_duplicate_rows_including_id',
    'exact_duplicate_rows_excluding_id', 'duplicate_profile_groups',
    'rows_in_duplicate_profile_groups', 'cross_split_duplicate_profile_groups',
    'rows_in_cross_split_duplicate_profile_groups', 'missing_by_field'
]}

{'id_unique': True,
 'exact_duplicate_rows_including_id': 0,
 'exact_duplicate_rows_excluding_id': 35,
 'duplicate_profile_groups': 52,
 'rows_in_duplicate_profile_groups': 108,
 'cross_split_duplicate_profile_groups': 0,
 'rows_in_cross_split_duplicate_profile_groups': 0,
 'missing_by_field': {'ID': 0,
  'LIMIT_BAL': 0,
  'SEX': 0,
  'EDUCATION': 0,
  'MARRIAGE': 0,
  'AGE': 0,
  'PAY_0': 0,
  'PAY_2': 0,
  'PAY_3': 0,
  'PAY_4': 0,
  'PAY_5': 0,
  'PAY_6': 0,
  'BILL_AMT1': 0,
  'BILL_AMT2': 0,
  'BILL_AMT3': 0,
  'BILL_AMT4': 0,
  'BILL_AMT5': 0,
  'BILL_AMT6': 0,
  'PAY_AMT1': 0,
  'PAY_AMT2': 0,
  'PAY_AMT3': 0,
  'PAY_AMT4': 0,
  'PAY_AMT5': 0,
  'PAY_AMT6': 0,
  'DEFAULT_NEXT_MONTH': 0}}

The leakage-control key uses all 23 original predictors and excludes both ID and target. It produces 29,944 groups; 52 groups are repeated, covering 108 rows. Automated validation confirms that zero groups cross partitions. The separate count of 35 exact pairs excluding only ID is retained as a data-quality statistic.

## Target balance and split audit

In [4]:
print(summary['target_distribution_full_dataset_for_reporting'])
print(summary['split_balance_integrity_audit'])
print(summary['final_test_label_policy'])

[{'target': '0', 'count': 23364, 'percentage': 77.88}, {'target': '1', 'count': 6636, 'percentage': 22.12}]
[{'split': 'test', 'target': '0', 'count': 4673, 'split_total': 6000, 'percentage': 77.8833}, {'split': 'test', 'target': '1', 'count': 1327, 'split_total': 6000, 'percentage': 22.1167}, {'split': 'train', 'target': '0', 'count': 14018, 'split_total': 18000, 'percentage': 77.8778}, {'split': 'train', 'target': '1', 'count': 3982, 'split_total': 18000, 'percentage': 22.1222}, {'split': 'validation', 'target': '0', 'count': 4673, 'split_total': 6000, 'percentage': 77.8833}, {'split': 'validation', 'target': '1', 'count': 1327, 'split_total': 6000, 'percentage': 22.1167}]
Test labels used only to create and mechanically verify stratification; not used for exploratory feature or design decisions.


![Training target balance](../reports/figures/day2_training_target_balance.png)

## Undocumented categories and numerical checks

In [5]:
summary['undocumented_category_frequencies'], summary['numeric_ranges'], summary['anomaly_counts']

([{'field': 'EDUCATION', 'value': '0', 'count': 14, 'documented': False},
  {'field': 'EDUCATION', 'value': '5', 'count': 280, 'documented': False},
  {'field': 'EDUCATION', 'value': '6', 'count': 51, 'documented': False},
  {'field': 'MARRIAGE', 'value': '0', 'count': 54, 'documented': False},
  {'field': 'PAY_0', 'value': '-2', 'count': 2759, 'documented': False},
  {'field': 'PAY_0', 'value': '0', 'count': 14737, 'documented': False},
  {'field': 'PAY_2', 'value': '-2', 'count': 3782, 'documented': False},
  {'field': 'PAY_2', 'value': '0', 'count': 15730, 'documented': False},
  {'field': 'PAY_3', 'value': '-2', 'count': 4085, 'documented': False},
  {'field': 'PAY_3', 'value': '0', 'count': 15764, 'documented': False},
  {'field': 'PAY_4', 'value': '-2', 'count': 4348, 'documented': False},
  {'field': 'PAY_4', 'value': '0', 'count': 16455, 'documented': False},
  {'field': 'PAY_5', 'value': '-2', 'count': 4546, 'documented': False},
  {'field': 'PAY_5', 'value': '0', 'count': 169

Undocumented codes are mapped to additional `Unknown/Other` labels without using the target; original numeric values remain intact. Negative bill amounts and large monetary extremes are flagged, not altered.

![Repayment codes](../reports/figures/day2_repayment_code_frequencies.png)

![Audit categories](../reports/figures/day2_audit_category_frequencies.png)

## Approved feature scope

In [6]:
features = primary_feature_names(rows[0].keys())
print(f"Primary features ({len(features)}): {features}")
print('Audit only: SEX, AGE, MARRIAGE, EDUCATION')

Primary features (19): ['LIMIT_BAL', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
Audit only: SEX, AGE, MARRIAGE, EDUCATION


Excluding the audit variables is a responsible choice for this portfolio demonstration. It does not guarantee fairness or regulatory compliance, and extensive fairness certification is outside the MVP.